# TP2 — Encodeur Transformer hiérarchique, version 2 : validation, **test**, attaques, sauvegarde sur Drive

**Pourquoi une version 2.** La première version (`tp2_encoder_colab.ipynb`) ne prédit que la **validation** et ne conserve pas les modèles. Pour évaluer l'encodeur au **jeu de test** (étape 7) et sous **attaque** (étape 8), il faut le réentraîner et prédire aussi le test. Cette version : (1) entraîne `distilbert-base-uncased` à deux têtes sur les **5 rotations** (mêmes réglages : 2 époques, taux 3e-5, lot 32, 256 jetons, graine 42) ; (2) prédit **validation et test**, avec Monte-Carlo dropout (10 passages) ; (3) prédit, pour la **rotation 1**, un échantillon de test **attaqué** (10 conditions), avec le texte brut **et** le texte à homoglyphes normalisés ; (4) **sauvegarde chaque modèle sur Google Drive** (`MyDrive/TP2_models/`), pour les réutiliser sans réentraîner.

**Zone de vérité.** Le notebook principal local contient les analyses. Ce notebook ne fait que calculer ; il n'utilise **jamais** de log de test pour entraîner ou régler quoi que ce soit.

**Mode d'emploi.** (1) GPU T4. (2) Cellule 2 : déposer `encoder_data_v2.csv.gz` et `encoder_attack_bundle.csv.gz` (dossier `TP/TP2/colab/`) ; la cellule vérifie les fichiers sur le disque et vous redemande ce qui manque. (3) Autoriser Google Drive. (4) *Tout exécuter* (environ 50 minutes ; reprendre avec `START`/`END`). (5) Décompresser `tp2_colab_v2_out.zip` dans `TP/TP2/colab/out_v2/`.

In [ ]:
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN : changer le type d\'exécution')

In [ ]:
import os
from google.colab import files
NEED = ['encoder_data_v2.csv.gz', 'encoder_attack_bundle.csv.gz']
def find(name):
    base = name.replace('.csv.gz', '')
    return [f for f in os.listdir('.') if f.startswith(base) and f.endswith('.gz')]
for _ in range(3):
    miss = [n for n in NEED if not find(n)]
    if not miss: break
    print('À déposer :', miss); files.upload()
for n in NEED:
    cands = find(n)
    assert cands, f'Fichier absent : {n}'
    if n not in cands: os.system(f'cp "{sorted(cands)[-1]}" {n}')      # Colab renomme les doublons : on retrouve le nom attendu
print('Fichiers présents :', sorted(os.listdir('.')))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/TP2_models'; os.makedirs(DRIVE, exist_ok=True); print('Dossier Drive :', DRIVE, os.listdir(DRIVE))

In [ ]:
%%writefile taxonomy_and_rotations_v1.json
{
 "taxonomy": {
  "authentication": [
   "authentication-failed",
   "authentication-success"
  ],
  "identity_session": [
   "user-creation",
   "user-deletion",
   "user-logout",
   "user-session-open"
  ],
  "network": [
   "connection-closed",
   "connection-failed",
   "connection-opened",
   "network-filtered",
   "network-traffic"
  ],
  "web": [
   "http-request-failure",
   "http-request-success"
  ],
  "file": [
   "directory-changed",
   "directory-created",
   "directory-deleted",
   "file-action-failure",
   "file-deleted",
   "file-modification",
   "file-read",
   "file-write"
  ],
  "process": [
   "process-ended",
   "process-error",
   "process-info",
   "process-shutdown",
   "process-started"
  ],
  "system": [
   "database-operation",
   "hardware-monitoring",
   "system-configuration-changed"
  ],
  "security_alert": [
   "ids-alert"
  ]
 },
 "rotations": [
  [
   "directory-created",
   "http-request-success",
   "process-info",
   "process-started",
   "system-configuration-changed",
   "user-session-open"
  ],
  [
   "database-operation",
   "file-deleted",
   "network-traffic",
   "process-ended",
   "process-error",
   "user-creation"
  ],
  [
   "authentication-failed",
   "connection-failed",
   "directory-changed",
   "file-action-failure",
   "http-request-failure",
   "network-filtered"
  ],
  [
   "file-modification",
   "file-read",
   "hardware-monitoring",
   "ids-alert",
   "process-shutdown",
   "user-logout"
  ],
  [
   "authentication-success",
   "connection-closed",
   "connection-opened",
   "directory-deleted",
   "file-write",
   "user-deletion"
  ]
 ],
 "seed": 42,
 "tries": 1,
 "classes_paper_order": [
  "authentication-failed",
  "authentication-success",
  "connection-closed",
  "connection-failed",
  "connection-opened",
  "database-operation",
  "directory-changed",
  "directory-created",
  "directory-deleted",
  "file-action-failure",
  "file-deleted",
  "file-modification",
  "file-read",
  "file-write",
  "hardware-monitoring",
  "http-request-failure",
  "http-request-success",
  "ids-alert",
  "network-filtered",
  "network-traffic",
  "process-ended",
  "process-error",
  "process-info",
  "process-shutdown",
  "process-started",
  "system-configuration-changed",
  "user-creation",
  "user-deletion",
  "user-logout",
  "user-session-open"
 ]
}

In [ ]:
%%writefile tp2_encoder.py
"""Encodeur Transformer hiérarchique pour logs en monde ouvert (TP2, étape 6). Calcul sur GPU (Colab ou Kaggle).

Une base (DistilBERT / BERT) + deux têtes linéaires : famille (F sorties) et classe fine (K sorties, classes CONNUES seulement).
Perte = entropie croisée famille + entropie croisée classe fine. À l'inférence : logits des deux têtes, et Monte-Carlo dropout
(T passages avec dropout actif) pour estimer l'incertitude. Les scores d'incertitude sont calculés ensuite, en local.
"""
import json
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup


def pick_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class HierEncoder(nn.Module):
    def __init__(self, name, n_fine, n_fam, dropout=0.1):
        super().__init__()
        self.base = AutoModel.from_pretrained(name)
        h = self.base.config.hidden_size
        self.drop = nn.Dropout(dropout); self.fam = nn.Linear(h, n_fam); self.fine = nn.Linear(h, n_fine)

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float(); pooled = (out * m).sum(1) / m.sum(1)      # moyenne des jetons, remplissage exclu
        z = self.drop(pooled)
        return self.fam(z), self.fine(z)


def _batch(ids, idx, pad_id, device):
    L = max(len(ids[j]) for j in idx)
    x = torch.full((len(idx), L), pad_id, dtype=torch.long); m = torch.zeros((len(idx), L), dtype=torch.long)
    for r, j in enumerate(idx):
        x[r, :len(ids[j])] = torch.tensor(ids[j]); m[r, :len(ids[j])] = 1
    return x.to(device), m.to(device)


def train_encoder(name, texts, y_fine, y_fam, n_fine, n_fam, *, max_len=256, epochs=2, lr=3e-5, batch_size=32, seed=42, device=None, warmup=0.06):
    device = device or pick_device(); set_seed(seed)
    tok = AutoTokenizer.from_pretrained(name); model = HierEncoder(name, n_fine, n_fam).to(device)
    ids = tok(list(texts), truncation=True, max_length=max_len)["input_ids"]
    yf = torch.tensor(np.asarray(y_fine), dtype=torch.long); yfa = torch.tensor(np.asarray(y_fam), dtype=torch.long)
    total = math.ceil(len(ids) / batch_size) * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01); sched = get_linear_schedule_with_warmup(opt, int(warmup * total), total)
    ce = nn.CrossEntropyLoss(); rng = np.random.default_rng(seed); step = 0; t0 = time.perf_counter(); hist = []
    for ep in range(epochs):
        model.train(); perm = rng.permutation(len(ids)); run = 0.0
        for s in range(0, len(perm), batch_size):
            idx = perm[s:s + batch_size]; x, m = _batch(ids, idx, tok.pad_token_id, device)
            lf, lc = model(x, m); loss = ce(lf, yfa[idx].to(device)) + ce(lc, yf[idx].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1; run += float(loss.item())
            if step % 100 == 0: hist.append({"step": step, "loss": run / 100, "min": round((time.perf_counter() - t0) / 60, 2)}); run = 0.0
    info = {"model": name, "max_len": max_len, "epochs": epochs, "lr": lr, "batch_size": batch_size, "seed": seed, "device": device, "steps": step,
            "train_minutes": round((time.perf_counter() - t0) / 60, 2), "n_train": len(ids), "history": hist}
    return model, tok, info


@torch.no_grad()
def predict(model, tok, texts, *, max_len=256, batch_size=128, mc=10, device=None):
    device = device or pick_device(); model.to(device)
    ids = tok(list(texts), truncation=True, max_length=max_len)["input_ids"]; order = np.argsort([len(x) for x in ids])
    n = len(ids); nf = model.fine.out_features; nfa = model.fam.out_features
    out = {"logits_fine": np.zeros((n, nf), np.float32), "logits_fam": np.zeros((n, nfa), np.float32), "mc_fine": np.zeros((n, nf), np.float32), "mc_fam": np.zeros((n, nfa), np.float32)}
    for s in range(0, n, batch_size):
        idx = order[s:s + batch_size]; x, m = _batch(ids, idx, tok.pad_token_id, device)
        model.eval(); lf, lc = model(x, m); out["logits_fam"][idx] = lf.float().cpu().numpy(); out["logits_fine"][idx] = lc.float().cpu().numpy()
        if mc:
            model.train()                                                                    # dropout actif : T passages stochastiques
            pf = torch.zeros_like(lf); pc = torch.zeros_like(lc)
            for _ in range(mc):
                a, b = model(x, m); pf += torch.softmax(a.float(), -1); pc += torch.softmax(b.float(), -1)
            out["mc_fam"][idx] = (pf / mc).cpu().numpy(); out["mc_fine"][idx] = (pc / mc).cpu().numpy(); model.eval()
    return out

In [ ]:
import json, time, numpy as np, pandas as pd, torch
import tp2_encoder as E
df = pd.read_csv('encoder_data_v2.csv.gz'); df['text'] = df.text.fillna(''); tax = json.load(open('taxonomy_and_rotations_v1.json'))
tr = df[df.split == 'train'].reset_index(drop=True); va = df[df.split == 'val'].reset_index(drop=True); te = df[df.split == 'test'].reset_index(drop=True)
att = pd.read_csv('encoder_attack_bundle.csv.gz'); att['text'] = att.text.fillna(''); att['text_unconf'] = att.text_unconf.fillna('')
FAMS = sorted(tax['taxonomy']); PAPER = tax['classes_paper_order']; ROT = tax['rotations']
print(len(tr), 'entraînement |', len(va), 'validation |', len(te), 'test |', len(att), 'lignes attaquées')
os.makedirs('/content/out_v2', exist_ok=True)

In [ ]:
START, END = 0, 5          # rotations à traiter (0 à 4)
NAME, MAXLEN = 'distilbert-base-uncased', 256
for rot in range(START, END):
    tag = f'uncased_rot{rot + 1}_len{MAXLEN}_s42'
    if os.path.exists(f'/content/out_v2/{tag}_test.npz'): print('déjà fait :', tag); continue
    hidden = set(ROT[rot]); known = [c for c in PAPER if c not in hidden]; c2i = {c: i for i, c in enumerate(known)}; f2i = {f: i for i, f in enumerate(FAMS)}
    mdir = f'{DRIVE}/{tag}'; t0 = time.time(); tk_known = tr[tr.category.isin(known)].reset_index(drop=True)
    if os.path.exists(mdir + '/model.pt'):
        tok = E.AutoTokenizer.from_pretrained(NAME); model = E.HierEncoder(NAME, len(known), len(FAMS)).to(E.pick_device()); model.load_state_dict(torch.load(mdir + '/model.pt', map_location='cpu')); info = json.load(open(mdir + '/info.json')); print(tag, ': modèle chargé depuis Drive')
    else:
        model, tok, info = E.train_encoder(NAME, tk_known.text.values, tk_known.category.map(c2i).values, tk_known.family.map(f2i).values, len(known), len(FAMS), max_len=MAXLEN, epochs=2, lr=3e-5, batch_size=32, seed=42)
        os.makedirs(mdir, exist_ok=True); torch.save(model.state_dict(), mdir + '/model.pt'); json.dump(info, open(mdir + '/info.json', 'w'))
    for nm, d in (('val', va), ('test', te)):
        out = E.predict(model, tok, d.text.values, max_len=MAXLEN, mc=10); np.savez_compressed(f'/content/out_v2/{tag}_{nm}.npz', **out)
    if rot == 0:
        for col in ('text', 'text_unconf'):
            out = E.predict(model, tok, att[col].values, max_len=MAXLEN, mc=0); np.savez_compressed(f'/content/out_v2/{tag}_attack_{col}.npz', **out)
    json.dump({'tag': tag, 'known': known, 'families': FAMS, 'hidden': sorted(hidden), 'info': info}, open(f'/content/out_v2/{tag}.json', 'w'), indent=1)
    acc = (np.load(f'/content/out_v2/{tag}_test.npz')['logits_fine'].argmax(1) == te.category.map(lambda c: c2i.get(c, -1)).values)[te.category.isin(known)].mean()
    print(f"{tag}: {info['train_minutes']} min ; exactitude fermée sur les connus (test) {acc:.3f} ; total {time.time() - t0:.0f} s"); del model; torch.cuda.empty_cache()

In [ ]:
!cd /content && zip -qr tp2_colab_v2_out.zip out_v2
from google.colab import files
files.download('/content/tp2_colab_v2_out.zip')